In [2]:
pip install scikit-learn

   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/8.2 MB 7.5 MB/s eta 0:00:01
   --------------- ------------------------ 3.1/8.2 MB 8.7 MB/s eta 0:00:01
   ------------------------ --------------- 5.0/8.2 MB 9.1 MB/s eta 0:00:01
   ---------------------------------- ----- 7.1/8.2 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 8.8 MB/s  0:00:01
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   -- ------------------------------------- 1.8/36.6 MB 10.2 MB/s eta 0:00:04
   ---- ----------------------------------- 3.7/36.6 MB 9.0 MB/s eta 0:00:04
   ----- ---------------------------------- 5.2/36.6 MB 9.4 MB/s eta 0:00:04
   -------- ------------------------------- 7.6/36.6 MB 9.3 MB/s eta 0:00:04
   ---------- ----------------------------- 10.0/36.6 MB 9.8 MB/s eta 0:00:03
   ------------- -------------------------- 12.3/36.6 MB 10.2 MB/s eta 0:00:03
   ---------------

In [4]:
df = pd.read_csv(
    r"C:\City Bike\data\processed\citibike_all_data.csv"
)

df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
df["hour"] = df["started_at"].dt.hour
df["is_weekend"] = df["started_at"].dt.dayofweek >= 5

features = [
    "trip_distance_km",
    "hour",
    "is_weekend",
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
    "member_casual",
]

target = "trip_duration_min"

model_df = df[features + [target]].dropna()

C:\Users\Chinmaya Harekrishna\AppData\Local\Temp\ipykernel_33784\1911560673.py:1: DtypeWarning: Columns (5: start_station_id, 7: end_station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [6]:
print(df.columns.tolist())

['ride_id', 'rideable_type', 'started_at', 'ended_at', 'trip_duration_min', 'trip_distance_km', 'member_casual']


In [13]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the complete dataset
df = pd.read_csv(
    r"C:\City Bike\data\processed\citibike_all_data.csv",
    low_memory=False,
)

# Convert timestamps and create numeric time features
df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
df["ended_at"] = pd.to_datetime(df["ended_at"], errors="coerce")
df["start_year"] = df["started_at"].dt.year
df["start_month"] = df["started_at"].dt.month
df["hour"] = df["started_at"].dt.hour
df["day_of_week"] = df["started_at"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# Exclude the target, raw timestamps, IDs, and merge-only helper columns.
target = "trip_duration_min"
excluded_columns = {
    target,
    "ride_id",
    "started_at",
    "ended_at",
    "date_hour",
    "time",
}

feature_columns = [
    column for column in df.columns
    if column not in excluded_columns and df[column].notna().any()
]

model_df = df[feature_columns + [target]].dropna(subset=[target])
X = model_df[feature_columns]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number", "bool"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ],
    remainder="drop",
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression()),
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("Features used:", len(feature_columns))
print("Rows used:", len(model_df))
print("MAE:", mae)
print("RMSE:", rmse)
print("R2 score:", r2)

Features used: 44
Rows used: 5244973
MAE: 5.360800585038741
RMSE: 17.5249504115912
R2 score: 0.21038468654440334


In [9]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score